# PreBuilt Middleware Part 2

| Middleware            | Simple meaning                                     | Example                                | Real scenario                                  | When to use                             | When NOT to use                                           |
| --------------------- | -------------------------------------------------- | -------------------------------------- | ---------------------------------------------- | --------------------------------------- | --------------------------------------------------------- |
| **Retry**             | Re-run a failed operation                          | API fails → retry                      | Payment/status API temporarily returns **503** | Temporary/transient failures            | Permanent errors like invalid API key                     |
| **Human-in-the-loop** | Pause and ask a human                              | Agent wants to delete data → approval  | Banking/refund/customer account actions        | High-risk actions                       | Low-risk read-only tools                                  |
| **Model fallback**    | Switch to another model                            | GPT fails → Claude                     | Production chatbot during provider outage      | Reliability                             | When models have incompatible output schemas              |
| **Summarization**     | Compress old context                               | 50 messages → summary                  | Long-running support agent                     | Context window/token control            | When every original message is critical                   |
| **Model call limit**  | Limit how many times the LLM can be called         | Max 10 model calls → stop              | Research agent stuck in a loop                 | Cost control, preventing runaway agents | Complex workflows that legitimately need many model calls |
| **Tool call limit**   | Limit how many times tools can be called           | Search tool → max 10 calls             | Research agent repeatedly searching the web    | API/rate-limit/cost control             | Tasks where tool calls naturally vary significantly       |
| **PII detection**     | Detect and protect personal information            | Email → `[REDACTED_EMAIL]`             | Healthcare/customer-support agent              | Privacy and compliance                  | When the agent legitimately needs the information         |
| **Custom PII types**  | Detect your organization's specific sensitive data | `EMP-12345` → `[REDACTED_EMPLOYEE_ID]` | Enterprise HR agent                            | Company-specific IDs/secrets            | When the pattern is not actually sensitive                |
| **To-do list**        | Give the agent a task checklist                    | Research → search → analyze → write    | Coding/research agent with many steps          | Complex multi-step tasks                | Simple one-step questions                                 |
| **LLM tool selector** | Select only relevant tools from many tools         | 100 tools → select 5 relevant tools    | Enterprise agent with many APIs/MCP tools      | Large toolsets, cost/context reduction  | Only 3–5 tools available                                  |


# TO Do List

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
llm="groq:llama-3.1-8b-instant"

In [3]:
# --- Core LangChain ---
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.tools import tool as tool_rt, ToolRuntime

In [2]:
# --- LangGraph (checkpointing, resuming) ---
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

In [4]:
@tool
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie at the cinema."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM only",
        "oppenheimer": "Sold out for tonight",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

In [5]:
@tool
def book_seats(movie_title: str, seat_count: int) -> str:
    """Book seats for a movie. Irreversible once confirmed."""
    return f"Booked {seat_count} seat(s) for {movie_title}."

In [6]:
@tool
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."


In [7]:
@tool
def check_order_status(booking_id: str) -> str:
    """Check the status of an existing booking."""
    return f"Booking {booking_id}: confirmed, 2 seats, Interstellar, 7:00 PM."

In [8]:
@tool
def get_refund_policy() -> str:
    """Get the cinema's refund policy -- exact wording, not to be paraphrased."""
    return "Refunds available up to 2 hours before showtime. No refunds after that."

In [9]:

@tool
def lookup_seat_map(movie_title: str, seat_number: str) -> str:
    """Look up a specific seat -- fails if the seat number format is wrong."""
    if not seat_number or not seat_number[0].isalpha():
        raise ValueError(f"Malformed seat number '{seat_number}' -- expected a letter+number like 'A12'.")
    return f"Seat {seat_number} for {movie_title}: available."

In [10]:
cinebot_tools = [check_showtimes, book_seats, cancel_booking, check_order_status, get_refund_policy, lookup_seat_map]

In [17]:
todo_agent = create_agent(
    model=llm,
    tools=cinebot_tools,
    middleware=[TodoListMiddleware()],
    system_prompt="You are CineBot, a helpful assistant for managing movie bookings and showtimes."

)

In [18]:
result = todo_agent.invoke({
    "messages": [("user", "I want to plan a movie night: check what's showing, pick something good, and book 2 seats.")]
})

In [20]:
from rich import print

In [21]:
print(result)

{
    'messages': [
        HumanMessage(
            content="I want to plan a movie night: check what's showing, pick something good, and book 2 seats.",
            additional_kwargs={},
            response_metadata={},
            id='91e803dd-71d3-4849-9143-e02b488a3fb5'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'tool_calls': [
                    {
                        'id': 'egb2fcg12',
                        'function': {
                            'arguments': '{"todos":[{"content":"Check movie 
showtimes","status":"pending"},{"content":"Choose a movie","status":"pending"},{"content":"Book 2 seats for the 
chosen movie","status":"pending"}]}',
                            'name': 'write_todos'
                        },
                        'type': 'function'
                    },
                    {
                        'id': 's3tkj7yn9',
                        'function': {
                            'arguments': '{"movie_title":"The latest Marvel movie"}',
                            'name': 'check_showtimes'
                        },
                        'type': 'function'
                    },
                    {
                        'id': '5hfw6wazx',
                        'function': {
                            'arguments': '{"movie_title":"The latest Marvel movie","seat_number":"F-12"}',
                            'name': 'lookup_seat_map'
                        },
                        'type': 'function'
                    },
                    {
                        'id': '3x20zr3s8',
                        'function': {
                            'arguments': '{"movie_title":"The latest Marvel movie","seat_number":"F-13"}',
                            'name': 'lookup_seat_map'
                        },
                        'type': 'function'
                    },
                    {
                        'id': 'qyhh5e0bt',
                        'function': {
                            'arguments': '{"movie_title":"The latest Marvel movie","seat_count":2}',
                            'name': 'book_seats'
                        },
                        'type': 'function'
                    },
                    {
                        'id': '2e74hjrkk',
                        'function': {'arguments': '{"booking_id":"B12345"}', 'name': 'check_order_status'},
                        'type': 'function'
                    }
                ]
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 188,
                    'prompt_tokens': 2665,
                    'total_tokens': 2853,
                    'completion_time': 0.243982292,
                    'completion_tokens_details': None,
                    'prompt_time': 0.191057738,
                    'prompt_tokens_details': None,
                    'queue_time': 0.335030621,
                    'total_time': 0.43504003
                },
                'model_name': 'llama-3.1-8b-instant',
                'system_fingerprint': 'fp_6a1eabf260',
                'service_tier': 'on_demand',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'groq'
            },
            id='lc_run--019feb08-175c-7452-8694-9f3864a095bf-0',
            tool_calls=[
                {
                    'name': 'write_todos',
                    'args': {
                        'todos': [
                            {'content': 'Check movie showtimes', 'status': 'pending'},
                            {'content': 'Choose a movie', 'status': 'pending'},
                            {'content': 'Book 2 seats for the chosen movie', 'status': 'pending'}
                        ]
                    },
                    'id': 'egb2fcg12',
                    'type': 'tool_call'
                },
                {
             

# LLM Tool Selector

In [22]:
for tool in cinebot_tools:
  print (tool.name)

check_showtimes

book_seats

cancel_booking

check_order_status

get_refund_policy

lookup_seat_map

In [23]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def show_tools(request, handler):
    print("\nTOOLS SENT TO MODEL:")
    print([tool.name for tool in request.tools])

    return handler(request)

In [24]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware


In [25]:
selector_agent = create_agent(
    model=llm,
    tools=cinebot_tools,
    middleware=[
        LLMToolSelectorMiddleware(
            model=llm,     # can be a CHEAPER model than the main agent
            max_tools=2,
            always_include=["check_showtimes"],  # always kept, doesn't count against max_tools
        ),
        show_tools
    ],
)

In [26]:
result = selector_agent.invoke({"messages": [("user", "Can you cancel my booking with ID B1234?")]})

TOOLS SENT TO MODEL:

['cancel_booking', 'check_order_status', 'check_showtimes']

TOOLS SENT TO MODEL:

['cancel_booking', 'check_order_status', 'check_showtimes']

In [27]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Can you cancel my booking with ID B1234?',
            additional_kwargs={},
            response_metadata={},
            id='7a9115cd-9b01-409a-b348-556d6c046084'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'tool_calls': [
                    {
                        'id': 'rtb2kcgcx',
                        'function': {'arguments': '{"booking_id":"B1234"}', 'name': 'cancel_booking'},
                        'type': 'function'
                    }
                ]
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 18,
                    'prompt_tokens': 349,
                    'total_tokens': 367,
                    'completion_time': 0.022246636,
                    'completion_tokens_details': None,
                    'prompt_time': 0.0253049,
                    'prompt_tokens_details': None,
                    'queue_time': 0.054261762,
                    'total_time': 0.047551536
                },
                'model_name': 'llama-3.1-8b-instant',
                'system_fingerprint': 'fp_4387d3edbb',
                'service_tier': 'on_demand',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'groq'
            },
            id='lc_run--019feb0a-e775-7981-afcf-d8da22775d06-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B1234'},
                    'id': 'rtb2kcgcx',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={'input_tokens': 349, 'output_tokens': 18, 'total_tokens': 367}
        ),
        ToolMessage(
            content='Booking B1234 cancelled.',
            name='cancel_booking',
            id='ea98de81-fa83-458d-a43e-9fbf730423ae',
            tool_call_id='rtb2kcgcx'
        ),
        AIMessage(
            content="However, since the function call was successful, I must follow the format exactly as 
requested. \n\nPlease note, however, this is a simulated response and not the actual response from the provided 
function.\n\nHowever, in this case, 'booking_id' was a required parameter, and it was specified correctly.",
            additional_kwargs={},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 61,
                    'prompt_tokens': 383,
                    'total_tokens': 444,
                    'completion_time': 0.139892921,
                    'completion_tokens_details': None,
                    'prompt_time': 0.036826338,
                    'prompt_tokens_details': None,
                    'queue_time': 0.055809801,
                    'total_time': 0.176719259
                },
                'model_name': 'llama-3.1-8b-instant',
                'system_fingerprint': 'fp_4387d3edbb',
                'service_tier': 'on_demand',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'groq'
            },
            id='lc_run--019feb0a-e96b-7b53-9569-40aa317702ad-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={'input_tokens': 383, 'output_tokens': 61, 'total_tokens': 444}
        )
    ]
}

# Tool Error

In [28]:
import langchain

print(langchain.__version__)

1.3.14

In [29]:
from langchain.agents.middleware import ToolErrorMiddleware

In [34]:
from langchain_core.tools import tool
import re


@tool
def lookup_seat_map(movie_title: str, seat_number: str) -> str:
    """Look up a specific seat — fails if the seat number format is wrong."""

    if not re.fullmatch(r"[A-Za-z]\d+", seat_number):
        raise ValueError(
            f"Malformed seat number '{seat_number}' — "
            "expected a letter+number like 'A12'."
        )

    return f"Seat {seat_number} for {movie_title}: available."

In [35]:
def on_seat_error(exc: Exception, request) -> str | None:
    if isinstance(exc, ValueError):
        # Return the EXCEPTION TYPE, not str(exc) -- internal detail never reaches the model
        return f"`{request.tool_call['name']}` failed with {type(exc).__name__}. Please provide a valid seat number like 'A12'."
    return None  # anything else propagates and halts the run


In [36]:
error_handled_agent = create_agent(
    model=llm,
    tools=cinebot_tools,
    # middleware=[ToolErrorMiddleware(on_error=on_seat_error)],
)

In [37]:
result = error_handled_agent.invoke({"messages": [("user", "Look up seat 12 for Dune Part Two")]})

ValueError: Malformed seat number '12' -- expected a letter+number like 'A12'.

In [39]:
error_handled_agent = create_agent(
    model=llm,
    tools=cinebot_tools,
    middleware=[ToolErrorMiddleware(on_error=on_seat_error)],
)

In [40]:
result = error_handled_agent.invoke({"messages": [("user", "Look up seat 12 for Dune Part Two")]})

KeyboardInterrupt: 

# Tool Retry

In [41]:
from langchain.agents.middleware import ToolRetryMiddleware

import random
random.random()

0.25789011884998514

In [42]:
import random
@tool
def flaky_showtime_check(movie_title: str) -> str:
    """Check showtimes via an external service that can transiently fail."""
    if not random.random() > 1:
        print("Facing Connection Error")
        raise ConnectionError("Simulated network failure -- exactly what a real external call risks.")
    return f"{movie_title}: showing at 8:00 PM."


In [45]:
resilient_tool_agent = create_agent(
    model=llm,
    tools=[flaky_showtime_check],
    middleware=[
        ToolRetryMiddleware(max_retries=3, backoff_factor=2.0, initial_delay=1.0, on_failure="continue"),
    ],

)

In [46]:
result = resilient_tool_agent.invoke({"messages": [("user", "Check showtimes for Interstellar")]})

Facing Connection Error

Facing Connection Error

Facing Connection Error

Facing Connection Error

Facing Connection Error

Facing Connection Error

Facing Connection Error

Facing Connection Error

BadRequestError: Error code: 400 - {'error': {'message': "tool call validation failed: attempted to call tool 'brave_search' which was not in request.tools", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=brave_search>{"query": "showtimes Interstellar"}</function>'}}

In [47]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Can you cancel my booking with ID B1234?',
            additional_kwargs={},
            response_metadata={},
            id='7a9115cd-9b01-409a-b348-556d6c046084'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'tool_calls': [
                    {
                        'id': 'rtb2kcgcx',
                        'function': {'arguments': '{"booking_id":"B1234"}', 'name': 'cancel_booking'},
                        'type': 'function'
                    }
                ]
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 18,
                    'prompt_tokens': 349,
                    'total_tokens': 367,
                    'completion_time': 0.022246636,
                    'completion_tokens_details': None,
                    'prompt_time': 0.0253049,
                    'prompt_tokens_details': None,
                    'queue_time': 0.054261762,
                    'total_time': 0.047551536
                },
                'model_name': 'llama-3.1-8b-instant',
                'system_fingerprint': 'fp_4387d3edbb',
                'service_tier': 'on_demand',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'groq'
            },
            id='lc_run--019feb0a-e775-7981-afcf-d8da22775d06-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B1234'},
                    'id': 'rtb2kcgcx',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={'input_tokens': 349, 'output_tokens': 18, 'total_tokens': 367}
        ),
        ToolMessage(
            content='Booking B1234 cancelled.',
            name='cancel_booking',
            id='ea98de81-fa83-458d-a43e-9fbf730423ae',
            tool_call_id='rtb2kcgcx'
        ),
        AIMessage(
            content="However, since the function call was successful, I must follow the format exactly as 
requested. \n\nPlease note, however, this is a simulated response and not the actual response from the provided 
function.\n\nHowever, in this case, 'booking_id' was a required parameter, and it was specified correctly.",
            additional_kwargs={},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 61,
                    'prompt_tokens': 383,
                    'total_tokens': 444,
                    'completion_time': 0.139892921,
                    'completion_tokens_details': None,
                    'prompt_time': 0.036826338,
                    'prompt_tokens_details': None,
                    'queue_time': 0.055809801,
                    'total_time': 0.176719259
                },
                'model_name': 'llama-3.1-8b-instant',
                'system_fingerprint': 'fp_4387d3edbb',
                'service_tier': 'on_demand',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'groq'
            },
            id='lc_run--019feb0a-e96b-7b53-9569-40aa317702ad-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={'input_tokens': 383, 'output_tokens': 61, 'total_tokens': 444}
        )
    ]
}

In [48]:
initial_delay=1.0
backoff_factor=2.0

In [ ]:
#delay = initial_delay * backoff_factor^retry_number